# Jugando con Árboles de Decisión 🌳
**Objetivo:** conocer cómo un árbol de decisión divide el espacio de características, visualizar sus regiones de decisión y experimentar con sus hiperparámetros (profundidad y criterio).

**Estructura del cuaderno**
1. Función para graficar regiones de decisión (versión mejorada).
2. Carga y preparación de datos (Iris).
3. Entrenamiento y comparación de árboles con distinta profundidad.
4. Visualización interna del árbol y reglas en texto.
5. Importancia de características.
6. Sobreajuste: exactitud vs. profundidad.
7. Zona de juego interactiva.

## 1. Función `plot_decision_regions` (mejorada)

In [ ]:
# Importamos NumPy para el manejo de arreglos y mallas numéricas
import numpy as np
# Importamos matplotlib para generar las gráficas
import matplotlib.pyplot as plt
# ListedColormap permite crear un mapa de colores propio con una lista de colores
from matplotlib.colors import ListedColormap


def plot_decision_regions(X, y, classifier, test_idx=None, resolution=0.02, ax=None, title=None):
    """Grafica las regiones de decisión de un clasificador entrenado con 2 características.

    Parámetros
    ----------
    X : ndarray (n_muestras, 2)  -> características (solo las 2 que se graficarán)
    y : ndarray (n_muestras,)    -> etiquetas de clase
    classifier : modelo ya entrenado que tenga el método .predict()
    test_idx : índices (rango o lista) de X que pertenecen al conjunto de prueba
    resolution : tamaño del paso de la malla (más pequeño = más detalle, más lento)
    ax : eje de matplotlib donde dibujar (permite hacer subgráficas)
    title : título opcional de la gráfica
    """
    # Si no nos pasan un eje, usamos el actual; así la función sirve sola o en subgráficas
    if ax is None:
        ax = plt.gca()

    # Marcadores por clase (se evita 'x' porque no admite borde y genera advertencias)
    markers = ('o', 's', '^', 'v', 'D')
    # Colores por clase
    colors = ('red', 'blue', 'lightgreen', 'gray', 'cyan')
    # Obtenemos las clases distintas presentes en y
    classes = np.unique(y)

    # Validamos que haya suficientes colores/marcadores para todas las clases
    if len(classes) > len(colors):
        raise ValueError(f"Solo se soportan {len(colors)} clases y se recibieron {len(classes)}.")

    # Creamos el mapa de colores usando solo tantos colores como clases existan
    cmap = ListedColormap(colors[:len(classes)])

    # Límites del eje horizontal: mínimo y máximo de la 1.ª característica, con 1 unidad de margen
    x1_min, x1_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    # Límites del eje vertical: igual, pero con la 2.ª característica
    x2_min, x2_max = X[:, 1].min() - 1, X[:, 1].max() + 1

    # Construimos una malla de puntos que cubre todo el plano a graficar
    xx1, xx2 = np.meshgrid(np.arange(x1_min, x1_max, resolution),
                           np.arange(x2_min, x2_max, resolution))

    # Aplanamos la malla y la acomodamos como (n_puntos, 2) para que el modelo prediga cada punto
    grid = np.c_[xx1.ravel(), xx2.ravel()]
    # Predecimos la clase de cada punto de la malla
    Z = classifier.predict(grid)
    # Devolvemos las predicciones a la forma 2D de la malla
    Z = Z.reshape(xx1.shape)

    # Pintamos las regiones de decisión como áreas de color semitransparentes
    ax.contourf(xx1, xx2, Z, alpha=0.3, cmap=cmap)
    # Ajustamos los límites de los ejes al tamaño de la malla
    ax.set_xlim(xx1.min(), xx1.max())
    ax.set_ylim(xx2.min(), xx2.max())

    # Dibujamos las muestras reales, una clase a la vez
    for idx, cl in enumerate(classes):
        ax.scatter(x=X[y == cl, 0],            # coordenada x de las muestras de la clase cl
                   y=X[y == cl, 1],            # coordenada y de las muestras de la clase cl
                   alpha=0.8,                  # transparencia para ver puntos encimados
                   color=colors[idx],          # color propio de la clase
                   marker=markers[idx],        # forma propia de la clase
                   label=f'Clase {cl}',        # texto de la leyenda
                   edgecolor='black')          # borde negro para resaltar cada punto

    # Resaltamos las muestras de prueba (FUERA del ciclo, para dibujarlas una sola vez)
    if test_idx is not None:                   # "is not None" evita errores con arreglos/rangos
        ax.scatter(X[test_idx, 0], X[test_idx, 1],
                   facecolor='none',           # círculo hueco
                   edgecolor='black',          # contorno negro
                   alpha=1.0,                  # totalmente opaco
                   linewidth=1,                # grosor del contorno
                   marker='o',                 # forma circular
                   s=100,                      # tamaño del círculo
                   label='Conjunto de prueba') # texto de la leyenda

    # Si se indicó título, lo colocamos
    if title:
        ax.set_title(title)
    # Mostramos la leyenda de clases
    ax.legend(loc='upper left')

## 2. Datos: Iris (usaremos largo y ancho del pétalo)

In [ ]:
# Cargador del conjunto de datos Iris
from sklearn.datasets import load_iris
# Función para dividir en entrenamiento y prueba
from sklearn.model_selection import train_test_split

# Cargamos Iris (150 flores, 3 especies, 4 medidas)
iris = load_iris()
# Nos quedamos con las columnas 2 y 3: largo y ancho del pétalo (las más informativas)
X = iris.data[:, [2, 3]]
# Etiquetas: 0 = setosa, 1 = versicolor, 2 = virginica
y = iris.target

# Dividimos 70 % entrenamiento / 30 % prueba; stratify mantiene la proporción de clases
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=1, stratify=y)

# Unimos ambos conjuntos solo para graficar todo junto
X_combined = np.vstack((X_train, X_test))
y_combined = np.hstack((y_train, y_test))
# Los índices de prueba son los que van después de los de entrenamiento
test_idx = range(len(y_train), len(y_combined))

# Mostramos tamaños para verificar la división
print("Entrenamiento:", X_train.shape, "| Prueba:", X_test.shape)
print("Especies:", iris.target_names.tolist())

## 3. Comparando árboles con distinta profundidad

In [ ]:
# Clasificador de árbol de decisión
from sklearn.tree import DecisionTreeClassifier

# Profundidades a probar (None = el árbol crece sin límite)
profundidades = [1, 2, 3, None]

# Creamos una figura de 2x2 subgráficas
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

# Recorremos cada profundidad junto con su eje
for depth, ax in zip(profundidades, axes.ravel()):
    # Creamos el árbol con el criterio gini y la profundidad máxima indicada
    tree = DecisionTreeClassifier(criterion='gini', max_depth=depth, random_state=1)
    # Entrenamos solo con los datos de entrenamiento
    tree.fit(X_train, y_train)
    # Exactitud en entrenamiento y en prueba
    acc_tr = tree.score(X_train, y_train)
    acc_te = tree.score(X_test, y_test)
    # Graficamos las regiones de decisión en el eje correspondiente
    plot_decision_regions(X_combined, y_combined, classifier=tree, test_idx=test_idx, ax=ax,
                          title=f"max_depth={depth} | train={acc_tr:.2f} | test={acc_te:.2f}")
    # Etiquetas de los ejes
    ax.set_xlabel('Largo del pétalo [cm]')
    ax.set_ylabel('Ancho del pétalo [cm]')

# Evitamos que los títulos se encimen
plt.tight_layout()
plt.show()

**Qué observar:** con `max_depth=1` el árbol solo hace un corte (subajuste). Al crecer la profundidad, las fronteras se vuelven rectangulares y más detalladas; sin límite, el árbol puede memorizar el entrenamiento.

## 4. Mirando dentro del árbol

In [ ]:
# plot_tree dibuja el árbol; export_text lo convierte en reglas legibles
from sklearn.tree import plot_tree, export_text

# Entrenamos un árbol de profundidad 3 para analizarlo
arbol = DecisionTreeClassifier(criterion='gini', max_depth=3, random_state=1)
arbol.fit(X_train, y_train)

# Dibujamos el árbol completo
plt.figure(figsize=(14, 7))
plot_tree(arbol,
          feature_names=['largo pétalo', 'ancho pétalo'],  # nombres de las características
          class_names=list(iris.target_names),             # nombres de las clases
          filled=True,                                      # colorea según la clase mayoritaria
          rounded=True)                                     # esquinas redondeadas
plt.show()

# Reglas del árbol en formato de texto (if / else)
print(export_text(arbol, feature_names=['largo_petalo', 'ancho_petalo']))

## 5. Importancia de características

In [ ]:
# feature_importances_ indica cuánto contribuyó cada variable a reducir la impureza
nombres = ['largo pétalo', 'ancho pétalo']
importancias = arbol.feature_importances_

# Gráfica de barras sencilla
plt.bar(nombres, importancias, color=['steelblue', 'orange'], edgecolor='black')
plt.ylabel('Importancia')
plt.title('Importancia de características (max_depth=3)')
plt.show()

# Imprimimos los valores numéricos
for n, i in zip(nombres, importancias):
    print(f"{n}: {i:.3f}")

## 6. Sobreajuste: exactitud vs. profundidad

In [ ]:
# Probamos profundidades de 1 a 10
rango = range(1, 11)
acc_train, acc_test = [], []   # listas para guardar resultados

for d in rango:
    t = DecisionTreeClassifier(max_depth=d, random_state=1).fit(X_train, y_train)  # entrenamos
    acc_train.append(t.score(X_train, y_train))   # exactitud en entrenamiento
    acc_test.append(t.score(X_test, y_test))      # exactitud en prueba

# Graficamos ambas curvas
plt.plot(rango, acc_train, 'o-', label='Entrenamiento')
plt.plot(rango, acc_test, 's--', label='Prueba')
plt.xlabel('max_depth')
plt.ylabel('Exactitud')
plt.title('¿Cuándo empieza el sobreajuste?')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 7. Zona de juego interactiva
Mueve los controles y observa cómo cambian las fronteras.

In [ ]:
# ipywidgets viene preinstalado en Colab y permite crear controles interactivos
from ipywidgets import interact, IntSlider, Dropdown

# La función se vuelve a ejecutar cada vez que se mueve un control
@interact(depth=IntSlider(min=1, max=10, step=1, value=3, description='Profundidad'),
          criterio=Dropdown(options=['gini', 'entropy'], value='gini', description='Criterio'),
          min_hoja=IntSlider(min=1, max=20, step=1, value=1, description='Min. hoja'))
def jugar(depth, criterio, min_hoja):
    # Entrenamos un árbol con los valores elegidos
    t = DecisionTreeClassifier(max_depth=depth, criterion=criterio,
                               min_samples_leaf=min_hoja, random_state=1).fit(X_train, y_train)
    # Graficamos regiones y mostramos la exactitud en prueba en el título
    plt.figure(figsize=(7, 5))
    plot_decision_regions(X_combined, y_combined, classifier=t, test_idx=test_idx,
                          title=f"depth={depth}, {criterio}, min_hoja={min_hoja} | test={t.score(X_test, y_test):.2f}")
    plt.xlabel('Largo del pétalo [cm]')
    plt.ylabel('Ancho del pétalo [cm]')
    plt.show()

### Retos para experimentar
1. ¿A partir de qué profundidad deja de mejorar la exactitud de prueba?
2. Cambia las columnas de `X` (por ejemplo `[0, 1]`, sépalo) y compara las regiones.
3. Sube `min_samples_leaf`: ¿las fronteras se vuelven más simples?
4. Compara `gini` contra `entropy`: ¿cambia algo?